# Módulo 2 — Detección de anomalías comerciales con One-Class SVM

## Caso comercial

Un equipo de ventas y revenue management necesita detectar transacciones atípicas para revisar descuentos fuera de política, operaciones con margen bajo, volúmenes inusuales y compras después de una recencia extrema.

El modelo aprenderá el comportamiento normal usando transacciones históricas revisadas. Después marcará operaciones nuevas que queden fuera de la frontera de normalidad. No se utiliza Isolation Forest ni Local Outlier Factor.

## Objetivos

- Construir datos comerciales sintéticos con unidades realistas.
- Explicar One-Class SVM y el kernel RBF.
- Visualizar las distribuciones y relaciones comerciales.
- Entrenar sólo con ventas normales.
- Evaluar precisión, recall, F1 y matriz de confusión.
- Interpretar las alertas y definir un plan productivo.

## 1. Contexto, problema y método

### Problema a resolver

Una venta con importe alto no necesariamente es problemática: puede corresponder a un cliente mayorista legítimo. De forma similar, un descuento elevado puede estar autorizado dentro de una campaña. La anomalía comercial aparece cuando la combinación completa es inusual: por ejemplo, muchas unidades, descuento alto, margen bajo y recencia extrema.

### Técnica elegida: One-Class SVM

One-Class SVM aprende una frontera alrededor de los datos normales. En vez de aprender la diferencia entre una clase normal y una clase anómala etiquetada, aprende qué región del espacio representa el comportamiento habitual.

Usaremos el kernel RBF para capturar relaciones no lineales entre importe, descuento, unidades, margen y recencia.

### Principios

El kernel RBF calcula similitud entre observaciones cercanas. La función de decisión es positiva para operaciones dentro de la frontera y negativa para operaciones fuera. En el notebook, una decisión negativa se convierte en alerta.

El parámetro nu representa una proporción esperada de observaciones fuera de la frontera. gamma controla el alcance de influencia de cada observación: un valor alto produce una frontera más flexible y un valor bajo una frontera más suave.

### Supuestos

El entrenamiento usa sólo operaciones normales. Las etiquetas anómalas se conservan únicamente para evaluar el ejercicio y no se entregan al modelo durante el ajuste.

## 1.1 Interpretación comercial de la frontera

Imaginemos cada transacción como un punto en un espacio de cinco dimensiones. Dos ventas pueden ser cercanas aunque sus importes sean diferentes si tienen descuentos, unidades, margen y recencia compatibles con su segmento.

One-Class SVM aprende una envolvente de normalidad. Una transacción fuera de esa envolvente recibe un score negativo. Cuanto más alejada esté de la frontera, mayor será su prioridad de revisión.

La técnica no determina si existe fraude, error o incumplimiento. Sólo indica que la combinación observada no se parece al patrón de referencia.

## 2. Configuración

Importamos herramientas para generar datos, transformar variables, entrenar One-Class SVM, calcular métricas y construir gráficas. La semilla permite repetir el experimento en Google Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('Entorno listo. Semilla:', RANDOM_STATE)

### Lectura ejecutiva de los resultados

La métrica más importante debe elegirse según la decisión comercial. Si el objetivo es no dejar pasar operaciones potencialmente problemáticas, el recall tiene prioridad: indica qué proporción de las anomalías conocidas fue recuperada. Si el equipo tiene poca capacidad de revisión, la precisión gana importancia: indica qué porcentaje de las alertas merece atención.

Un resultado con recall alto y precisión moderada puede ser adecuado como primera capa de monitoreo, porque captura casi todos los casos y deja que un analista descarte falsos positivos. En cambio, si la revisión es costosa o lenta, habría que elevar el umbral y aceptar que algunos casos no sean detectados.

### Cómo leer la matriz de confusión en términos de negocio

- **Verdadero positivo:** una transacción anómala fue alertada. Puede tratarse de descuento no autorizado, margen incorrecto, volumen excepcional o recencia inusual.
- **Falso positivo:** una transacción normal fue alertada. Puede ser una venta legítima de un cliente mayorista, una campaña autorizada o una operación en el borde del comportamiento normal.
- **Verdadero negativo:** una operación normal pasó sin alerta.
- **Falso negativo:** una anomalía conocida no fue alertada. Este es el riesgo más delicado si la revisión busca proteger margen o detectar abuso promocional.

La matriz permite traducir el resultado técnico a carga de trabajo. Si genera 119 alertas, el equipo debe saber si puede revisar 119 operaciones por periodo. El modelo no debe desplegarse sin relacionar el volumen de alertas con la capacidad operativa.

### Explicación detallada

numpy genera variables numéricas y controla la aleatoriedad. pandas crea la tabla de transacciones. StandardScaler es necesario porque importe, porcentaje, unidades y días tienen escalas diferentes. OneClassSVM aprende la región normal. Las funciones de métricas permiten cuantificar la calidad de las alertas.

## 3. Generación de datos comerciales

Construimos 1,200 operaciones normales y 60 operaciones anómalas. Las variables tienen interpretación directa:

- importe_venta_mxn: valor positivo de la operación.
- descuento_pct: porcentaje entre 0 y 100.
- unidades: cantidad entera vendida.
- margen_pct: margen comercial porcentual.
- dias_desde_ultima_compra: recencia positiva del cliente.

Las anomalías representan descuentos excepcionales, grandes volúmenes, importes inusuales, margen reducido y recencia extrema.

In [ ]:
n_normales, n_anomalias = 1200, 60
importe = rng.lognormal(4.1, 0.45, n_normales)
descuento = rng.beta(2.2, 7.0, n_normales) * 30
unidades = np.clip(rng.poisson(3.2, n_normales) + 1, 1, 20)
margen = np.clip(38 - 0.65*descuento - 0.35*unidades + rng.normal(0, 4, n_normales), 5, 55)
recencia = np.clip(rng.gamma(2.2, 12, n_normales), 1, 120)

normales = pd.DataFrame({
    'importe_venta_mxn': importe,
    'descuento_pct': descuento,
    'unidades': unidades,
    'margen_pct': margen,
    'dias_desde_ultima_compra': recencia,
    'es_anomalia_real': 0
})
anomalias = pd.DataFrame({
    'importe_venta_mxn': rng.lognormal(6.2, 0.55, n_anomalias),
    'descuento_pct': rng.uniform(45, 85, n_anomalias),
    'unidades': rng.integers(25, 100, n_anomalias),
    'margen_pct': rng.uniform(-8, 8, n_anomalias),
    'dias_desde_ultima_compra': rng.uniform(180, 600, n_anomalias),
    'es_anomalia_real': 1
})
df = pd.concat([normales, anomalias], ignore_index=True)
df.insert(0, 'transaccion_id', [f'V-{i:05d}' for i in range(1, len(df)+1)])
df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
variables = ['importe_venta_mxn', 'descuento_pct', 'unidades', 'margen_pct', 'dias_desde_ultima_compra']
print(f'Transacciones: {len(df):,} | Anomalías: {df.es_anomalia_real.sum():,}')
display(df.head())

### Explicación detallada de la generación

La distribución lognormal representa importes con cola derecha, algo frecuente en ventas. La beta produce descuentos concentrados en porcentajes bajos y moderados. Poisson es apropiada para cantidades enteras. El margen normal depende negativamente de descuento y unidades, reflejando presión comercial.

Las anomalías se generan con rangos extremos pero válidos para un escenario de revisión comercial. El margen puede ser negativo porque una operación puede venderse por debajo del costo o tener un dato de costo incorrecto; esto sí es una señal de negocio válida. La etiqueta real se usa sólo para evaluar.

## 4. Validación de calidad y exploración

Antes de modelar revisamos faltantes, estadísticos y reglas básicas de negocio. Los controles de calidad deben ejecutarse antes del detector para no confundir errores de captura con anomalías comerciales.

In [ ]:
print('Valores faltantes:')
print(df[variables].isna().sum())
display(df[variables].describe().round(2))
print('Controles:')
print({
    'importes_no_positivos': int((df.importe_venta_mxn <= 0).sum()),
    'descuentos_fuera_de_rango': int(((df.descuento_pct < 0) | (df.descuento_pct > 100)).sum()),
    'unidades_no_positivas': int((df.unidades <= 0).sum()),
    'recencias_negativas': int((df.dias_desde_ultima_compra < 0).sum())
})

### Interpretación de la validación

Los importes, unidades y días de recencia deben ser positivos. El descuento debe estar entre 0% y 100%. El margen no se restringe a valores positivos porque un margen bajo o negativo puede ser precisamente una señal comercial. Los estadísticos muestran las escalas y ayudan a detectar valores imposibles.

## 4.1 Visualizaciones de distribución

Los histogramas permiten observar concentración, asimetría y colas. Colorear por etiqueta sintética ayuda a auditar si las anomalías están construidas como casos comerciales distintos.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, variable in zip(axes.ravel(), variables):
    sns.histplot(data=df, x=variable, hue='es_anomalia_real', bins=35, kde=True,
                 palette={0:'#4C78A8', 1:'#E45756'}, alpha=.45, ax=ax)
    ax.set_title(variable)
axes.ravel()[-1].axis('off')
plt.tight_layout()
plt.show()

### Interpretación de las distribuciones

Se espera observar colas de importe y recencia. Las anomalías deben concentrarse en valores extremos, aunque una venta extrema no es automáticamente incorrecta. La revisión debe considerar cliente, vendedor, canal, campaña y autorización.

## 4.2 Relaciones comerciales

El mapa de dispersión ayuda a entender combinaciones: descuento versus margen, importe versus unidades y recencia versus importe.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
relaciones = [('descuento_pct', 'margen_pct'), ('importe_venta_mxn', 'unidades'), ('dias_desde_ultima_compra', 'importe_venta_mxn')]
for ax, (x, y) in zip(axes, relaciones):
    sns.scatterplot(data=df, x=x, y=y, hue='es_anomalia_real',
                    palette={0:'#4C78A8', 1:'#E45756'}, alpha=.65, ax=ax)
    ax.set_title(f'{x} vs {y}')
plt.tight_layout()
plt.show()

### Interpretación de relaciones

Una relación negativa entre descuento y margen es comercialmente esperable. Las anomalías pueden aparecer como puntos alejados de la nube: descuentos altos con margen muy bajo, muchas unidades con importe excepcional o compras muy tardías con importes inusuales. Estas relaciones son precisamente las que una frontera no lineal puede capturar.

## 4.3 Correlaciones y resumen por tipo

La matriz de correlación describe relaciones lineales, pero One-Class SVM con RBF también puede aprender patrones no lineales. El resumen por etiqueta muestra cómo difieren las poblaciones del benchmark.

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df[variables].corr(), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('Correlación entre variables comerciales')
plt.show()
display(df.groupby('es_anomalia_real')[variables].agg(['median', 'mean', 'max']).round(2))

## 5. Preparación y escalamiento

Separamos variables de entrada y estandarizamos. El escalamiento es indispensable para que importe, porcentaje, unidades y recencia tengan una influencia comparable en el kernel RBF.

In [ ]:
X = df[variables].copy()
y_real = df['es_anomalia_real'].to_numpy()
escalador = StandardScaler()
X_escalada = escalador.fit_transform(X)
X_entrenamiento = X_escalada[y_real == 0]
print('Matriz total:', X_escalada.shape)
print('Matriz de entrenamiento normal:', X_entrenamiento.shape)
print('Medias escaladas:', X_escalada.mean(axis=0).round(3))

### Explicación detallada del escalamiento

X excluye el identificador y la etiqueta. y_real se conserva fuera del entrenamiento sólo para evaluación. X_entrenamiento contiene exclusivamente operaciones normales, lo que simula un escenario en el que contamos con históricos revisados. StandardScaler calcula media y desviación; después transforma todo usando la misma referencia.

## 6. Entrenamiento con One-Class SVM

El modelo aprenderá la frontera normal usando sólo X_entrenamiento. Después aplicaremos esa frontera a todas las transacciones para medir qué tan bien encuentra las anomalías.

In [ ]:
modelo = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
modelo.fit(X_entrenamiento)
prediccion = modelo.predict(X_escalada)
df['prediccion_anomalia'] = (prediccion == -1).astype(int)
df['svm_score'] = -modelo.decision_function(X_escalada)
print(f'Alertas generadas: {df.prediccion_anomalia.sum():,} ({df.prediccion_anomalia.mean():.1%})')
display(df.sort_values('svm_score', ascending=False).head(10))

### Explicación detallada del entrenamiento

kernel RBF permite una frontera curva. gamma='scale' calcula una escala razonable a partir de los datos. nu=0.05 establece una expectativa inicial de 5% de operaciones fuera de patrón.

predict devuelve 1 para observaciones dentro de la frontera y -1 para fuera. La conversión a 0 y 1 hace el resultado más fácil de reportar. El score invertido ordena de mayor a menor rareza.

## 7. Evaluación de resultados

Calculamos precisión, recall, F1 y matriz de confusión. La exactitud global no debe ser la métrica principal porque la mayoría de las operaciones son normales.

In [ ]:
y_pred = df['prediccion_anomalia']
precision = precision_score(y_real, y_pred)
recall = recall_score(y_real, y_pred)
print(f'Precisión: {precision:.1%}')
print(f'Recall:    {recall:.1%}')
print(classification_report(y_real, y_pred, target_names=['normal', 'anomalía'], digits=3))
cm = confusion_matrix(y_real, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Pred. normal', 'Pred. anomalía'],
            yticklabels=['Real normal', 'Real anomalía'])
plt.title('Matriz de confusión')
plt.xlabel('Predicción'); plt.ylabel('Real'); plt.show()

### Lectura ejecutiva de los resultados

La métrica más importante debe elegirse según la decisión comercial. Si el objetivo es no dejar pasar operaciones potencialmente problemáticas, el recall tiene prioridad: indica qué proporción de las anomalías conocidas fue recuperada. Si el equipo tiene poca capacidad de revisión, la precisión gana importancia: indica qué porcentaje de las alertas merece atención.

Un resultado con recall alto y precisión moderada puede ser adecuado como primera capa de monitoreo, porque captura casi todos los casos y deja que un analista descarte falsos positivos. En cambio, si la revisión es costosa o lenta, habría que elevar el umbral y aceptar que algunos casos no sean detectados.

### Cómo leer la matriz de confusión en términos de negocio

- **Verdadero positivo:** una transacción anómala fue alertada. Puede tratarse de descuento no autorizado, margen incorrecto, volumen excepcional o recencia inusual.
- **Falso positivo:** una transacción normal fue alertada. Puede ser una venta legítima de un cliente mayorista, una campaña autorizada o una operación en el borde del comportamiento normal.
- **Verdadero negativo:** una operación normal pasó sin alerta.
- **Falso negativo:** una anomalía conocida no fue alertada. Este es el riesgo más delicado si la revisión busca proteger margen o detectar abuso promocional.

La matriz permite traducir el resultado técnico a carga de trabajo. Si genera 119 alertas, el equipo debe saber si puede revisar 119 operaciones por periodo. El modelo no debe desplegarse sin relacionar el volumen de alertas con la capacidad operativa.

### Interpretación detallada de las métricas

Precisión indica qué proporción de la cola de revisión corresponde a anomalías conocidas. Recall indica qué proporción de anomalías fueron recuperadas. Un falso negativo puede permitir una operación con descuento indebido; un falso positivo implica tiempo de revisión.

Los valores dependen del benchmark y de la semilla. No deben presentarse como una garantía productiva: sirven para entender el comportamiento del método antes de usar datos reales.

## 8. Interpretación comercial de alertas

Convertimos el score en una cola de revisión y agregamos una señal descriptiva. La señal no reemplaza el modelo ni prueba una causa; sólo ayuda a comenzar la investigación.

In [ ]:
alertas = df[df.prediccion_anomalia == 1].copy()
alertas['senal_comercial'] = np.select(
    [alertas.descuento_pct >= 45,
     alertas.margen_pct <= 5,
     alertas.unidades >= 25,
     alertas.dias_desde_ultima_compra >= 180],
    ['descuento fuera de política', 'margen muy bajo', 'volumen excepcional', 'recencia extrema'],
    default='combinación atípica'
)
display(alertas.sort_values('svm_score', ascending=False)[['transaccion_id'] + variables + ['svm_score', 'senal_comercial']].head(15).round(2))
display(alertas.senal_comercial.value_counts().rename_axis('señal').to_frame('alertas'))

### Interpretación del score de One-Class SVM

`svm_score` es una medida de alejamiento respecto de la frontera aprendida. Los valores altos representan mayor rareza dentro del conjunto evaluado. El score sirve para ordenar una cola de revisión, pero no debe interpretarse como probabilidad de fraude ni como porcentaje de pérdida.

Para explicar una alerta, conviene acompañar el score con los atributos originales. Por ejemplo, una operación con score alto, descuento de 70%, margen cercano a cero y 80 unidades tiene una historia comercial más accionable que el score aislado. La revisión debe contrastar autorización, campaña, vendedor, cliente y costo registrado.

### Qué significa el resultado para una decisión comercial

El modelo puede usarse como un sistema de priorización en tres niveles:

1. **Prioridad alta:** score más alejado de la frontera y señales simultáneas de descuento, margen o volumen.
2. **Prioridad media:** score atípico con una sola señal comercial; requiere validación contextual.
3. **Prioridad baja:** score cercano al umbral; puede revisarse por muestreo o utilizarse para mejorar el histórico.

Este enfoque evita convertir una señal estadística en una sanción automática y crea un proceso de aprendizaje: cada revisión humana puede registrar si la alerta correspondía a una excepción válida, error de datos o problema comercial real.

### Interpretación operativa

El analista debe revisar autorización de descuento, costo correcto, campaña vigente, identidad del cliente, vendedor y canal. Una venta con margen negativo puede ser válida en liquidación o incorrecta por un costo desactualizado. One-Class SVM prioriza; el proceso comercial decide.

## 9. Sensibilidad de nu y gamma

Probamos distintas fronteras para observar el trade-off entre cobertura y carga de revisión. La calibración debe considerar la capacidad del equipo y el costo de omitir operaciones problemáticas.

In [ ]:
resultados = []
for nu in [.03, .05, .10]:
    for gamma in ['scale', 0.05, 0.20]:
        m = OneClassSVM(kernel='rbf', gamma=gamma, nu=nu)
        m.fit(X_entrenamiento)
        p = (m.predict(X_escalada) == -1).astype(int)
        resultados.append({'nu': nu, 'gamma': str(gamma), 'alertas': p.sum(),
                           'precision': precision_score(y_real, p),
                           'recall': recall_score(y_real, p)})
sensibilidad = pd.DataFrame(resultados)
display(sensibilidad.style.format({'nu':'{:.0%}', 'precision':'{:.1%}', 'recall':'{:.1%}'}))

### Lectura profunda de la sensibilidad

La tabla de sensibilidad muestra que no existe una configuración universalmente óptima. `nu` controla principalmente cuántas observaciones quedan fuera de la frontera. Si aumenta, la cola de alertas crece; esto puede mejorar recall, pero también puede disminuir precisión.

`gamma` controla la flexibilidad del kernel RBF. Con gamma bajo, el modelo aprende una frontera más suave y general; con gamma alto, reacciona a estructuras más pequeñas y puede marcar como atípicos casos normales ubicados en zonas poco pobladas.

La selección recomendada debe basarse en una función de costo: costo de revisar una alerta falsa, costo de omitir una operación riesgosa y capacidad máxima de revisión. La mejor fila de la tabla no es necesariamente la de mayor recall, sino la que ofrece el mejor balance para el proceso comercial.

### Limitaciones de esta evaluación

Las anomalías fueron creadas artificialmente y son más extremas que muchas anomalías reales. Por ello, los resultados permiten estudiar el funcionamiento del algoritmo, pero no deben extrapolarse directamente a una operación productiva.

En producción se debe evaluar por periodos posteriores al entrenamiento, revisar cambios de temporada, separar campañas promocionales y medir estabilidad del volumen de alertas. También conviene comparar las alertas con pérdidas de margen, cancelaciones, autorizaciones y resultados de auditoría.

### Interpretación de sensibilidad

nu más alto suele producir más alertas y puede elevar recall, pero aumenta falsos positivos. gamma alto crea una frontera más flexible y puede capturar detalles, aunque con riesgo de sobreajuste. El valor final debe validarse por fecha y con resultados de revisión humana.

## 10. Conclusiones

- One-Class SVM es una alternativa a Isolation Forest y LOF para aprender la frontera de operaciones comerciales normales.
- Los datos usan unidades realistas: importes positivos, descuentos porcentuales, unidades enteras, margen y recencia.
- La detección depende de combinaciones, no únicamente de importes altos.
- El entrenamiento sólo con operaciones normales se aproxima mejor a un escenario real de monitoreo.
- Precisión, recall y capacidad de revisión deben definir el umbral operativo.
- Una alerta no prueba fraude, error o incumplimiento; requiere investigación contextual.

### Plan recomendado

1. Entrenar con ventas históricas revisadas y separarlas temporalmente.
2. Incorporar cliente, vendedor, canal, campaña y autorización.
3. Revisar las alertas con mayor score cada semana.
4. Registrar la resolución humana de cada alerta.
5. Recalibrar nu, gamma y variables con evidencia operativa.

## 11. Resumen reproducible

In [ ]:
print({'transacciones': len(df),
       'anomalias_reales': int(y_real.sum()),
       'alertas': int(y_pred.sum()),
       'precision': round(precision, 3),
       'recall': round(recall, 3)})